# 🐧 Day 01a: Linux Process Commands — ps, top, kill, cron, systemd

---

## 🎯 What You'll Master Today
- Process inspection: ps, top, htop
- Process control: kill, nice, nohup
- Job scheduling: cron, at, systemd timers
- Service management: systemctl

---

```
╔══════════════════════════════════════════════════════════════╗
║           LINUX PROCESS MANAGEMENT CHEATSHEET                ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  INSPECT                                                     ║
║    ps aux              — all processes, detailed             ║
║    ps -ef              — full format listing                 ║
║    top / htop          — real-time process monitor           ║
║    pgrep -f <pattern>  — find PID by name                   ║
║                                                              ║
║  CONTROL                                                     ║
║    kill <pid>          — send SIGTERM (graceful)             ║
║    kill -9 <pid>       — send SIGKILL (force)               ║
║    kill -STOP <pid>    — pause process                      ║
║    kill -CONT <pid>    — resume process                     ║
║    nice -n 10 cmd      — start with lower priority          ║
║    renice -n 5 -p pid  — change running process priority    ║
║    nohup cmd &         — keep running after logout          ║
║                                                              ║
║  SCHEDULE                                                    ║
║    crontab -e          — edit cron jobs                     ║
║    Format: min hour day month weekday command               ║
║    */5 * * * * cmd     — every 5 minutes                    ║
║    0 2 * * 0 cmd       — 2am every Sunday                   ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Linux Signals Reference
# ============================================

import signal

# Common signals every developer should know
signals = [
    ("SIGTERM", 15, "Graceful termination (default kill)", "Process can catch & cleanup"),
    ("SIGKILL", 9,  "Force kill (cannot be caught!)", "Kernel terminates immediately"),
    ("SIGINT",  2,  "Interrupt (Ctrl+C)", "Process can handle gracefully"),
    ("SIGSTOP", 19, "Pause process", "Cannot be caught (like SIGKILL)"),
    ("SIGCONT", 18, "Resume paused process", "Resumes from SIGSTOP"),
    ("SIGHUP",  1,  "Hangup / config reload", "Daemons reload config"),
    ("SIGUSR1", 10, "User-defined signal 1", "Custom application use"),
    ("SIGCHLD", 17, "Child process terminated", "Parent notified"),
    ("SIGPIPE", 13, "Broken pipe", "Writing to closed pipe/socket"),
    ("SIGSEGV", 11, "Segmentation fault", "Invalid memory access"),
]

print("  Linux Signals Cheatsheet")
print(f"  {'Signal':<10} {'#':<4} {'Description':<35} {'Behavior'}")
print("  " + "─" * 80)
for name, num, desc, behavior in signals:
    print(f"  {name:<10} {num:<4} {desc:<35} {behavior}")

print("\n  💡 Key rule: SIGKILL and SIGSTOP CANNOT be caught or ignored!")
print("  💡 Always try SIGTERM before SIGKILL — let process clean up.")

In [ ]:
# ============================================
# 2. Cron Expression Parser
# ============================================

def explain_cron(expression: str, command: str = "<command>"):
    """Parse and explain a cron expression."""
    parts = expression.split()
    if len(parts) != 5:
        print(f"  ❌ Invalid cron: need 5 fields, got {len(parts)}")
        return
    
    field_names = ["Minute", "Hour", "Day of Month", "Month", "Day of Week"]
    field_ranges = ["(0-59)", "(0-23)", "(1-31)", "(1-12)", "(0-7, 0=Sun)"]
    
    print(f"  Cron: {expression} {command}")
    print("  " + "─" * 50)
    
    for name, value, rng in zip(field_names, parts, field_ranges):
        meaning = "every" if value == "*" else value
        if value.startswith("*/"):
            meaning = f"every {value[2:]}" 
        print(f"    {name:<15} {rng:<15} = {value:<8} ({meaning})")


# Common cron patterns
cron_examples = [
    ("*/5 * * * *", "health_check.sh", "Every 5 minutes"),
    ("0 2 * * *", "backup.sh", "Daily at 2:00 AM"),
    ("0 0 * * 0", "weekly_report.py", "Weekly on Sunday midnight"),
    ("0 9 1 * *", "monthly_billing.py", "1st of every month at 9 AM"),
    ("30 */6 * * 1-5", "sync.sh", "Every 6 hours at :30, weekdays only"),
]

for expr, cmd, desc in cron_examples:
    print(f"\n  📅 {desc}")
    explain_cron(expr, cmd)
    print()

In [ ]:
# ============================================
# 3. systemd Service Anatomy
# ============================================

systemd_unit = """
# /etc/systemd/system/myapp.service

[Unit]
Description=My FastAPI Application
After=network.target          # Start after network is up
Wants=postgresql.service      # Prefer postgres running

[Service]
Type=simple                   # Main process = service
User=appuser                  # Don't run as root!
WorkingDirectory=/opt/myapp
ExecStart=/opt/myapp/venv/bin/uvicorn main:app --host 0.0.0.0 --port 8000
Restart=always                # Restart on crash
RestartSec=3                  # Wait 3s before restart
Environment=ENV=production

[Install]
WantedBy=multi-user.target    # Start on boot
"""

print("  systemd Service File Anatomy:")
print(systemd_unit)

# systemctl commands reference
commands = [
    ("systemctl start myapp",    "Start the service"),
    ("systemctl stop myapp",     "Stop the service"),
    ("systemctl restart myapp",  "Stop + start"),
    ("systemctl reload myapp",   "Reload config without restart"),
    ("systemctl status myapp",   "Check if running, see logs"),
    ("systemctl enable myapp",   "Auto-start on boot"),
    ("systemctl disable myapp",  "Remove from boot"),
    ("journalctl -u myapp -f",   "Stream live logs"),
]

print("  systemctl Quick Reference:")
print(f"  {'Command':<35} {'Description'}")
print("  " + "─" * 55)
for cmd, desc in commands:
    print(f"  {cmd:<35} {desc}")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | SIGTERM vs SIGKILL? | SIGTERM: graceful, catchable. SIGKILL: forced, uncatchable. Always try TERM first |
| 2 | What is a zombie process? | Child exited but parent hasn't called wait(). Shows as 'Z' in ps. Fix: kill parent |
| 3 | nohup vs disown vs &? | &: background. nohup: immune to SIGHUP. disown: remove from shell's job table |
| 4 | cron vs systemd timer? | cron: simple scheduling. systemd timer: more features (dependencies, logging, randomized delay) |
| 5 | How to find which process uses port 8080? | `lsof -i :8080` or `ss -tlnp \| grep 8080` or `netstat -tlnp \| grep 8080` |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • ps aux + top/htop for monitoring                      │
## │  • kill -TERM first, -KILL as last resort                │
## │  • cron: 5 fields (min hour dom month dow)               │
## │  • systemd: modern init, manages services + logging      │
## │  • Always run services as non-root user                  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **File Permissions** — chmod, chown, umask, setuid